In [39]:
# Card & Krueger (1994) Replication
# Phase 2: Deliverables & Submission Instructions

import pandas as pd
import numpy as np

col_names = [
    'sheet', 'chain', 'co_owned', 'state',
    'southj', 'centralj', 'northj', 'pa1', 'pa2', 'shore',
    'ncalls', 'empft', 'emppt', 'nmgrs', 'wage_st',
    'inctime', 'firstinc', 'bonus', 'pctaff', 'meals',
    'open', 'hrsopen', 'psoda', 'pfry', 'pentree',
    'nregs', 'nregs11',
    'type2', 'status2', 'date2', 'ncalls2',
    'empft2', 'emppt2', 'nmgrs2', 'wage_st2',
    'inctime2', 'firstinc2', 'special2', 'meals2',
    'open2r', 'hrsopen2', 'psoda2', 'pfry2', 'pentree2',
    'nregs2', 'nregs112'
]

DATA_URL = "https://raw.githubusercontent.com/Louisa328/DA-Midterm-Project-Minimum-Wages-and-Employment/main/data/raw/public.dat"
df = pd.read_csv(DATA_URL, sep=r'\s+', names=col_names, na_values='.')

print(f"Shape: {df.shape}")
print(f"NJ restaurants: {(df['state']==1).sum()}")
print(f"PA restaurants: {(df['state']==0).sum()}")
df.head()

# Construct FTE (Full-Time Equivalent)
df['fte'] = df['empft'] + df['nmgrs'] + 0.5 * df['emppt']
df['fte2'] = df['empft2'] + df['nmgrs2'] + 0.5 * df['emppt2']

print(df[['fte', 'fte2']].describe())

Shape: (410, 46)
NJ restaurants: 331
PA restaurants: 79
              fte        fte2
count  398.000000  396.000000
mean    20.998869   21.054293
std      9.749805    9.094453
min      5.000000    0.000000
25%     14.562500   14.500000
50%     19.500000   20.500000
75%     24.500000   26.500000
max     85.000000   60.500000


In [40]:
# Table 2 Full Replication
def calc_stats(df, var, state):
    x = df.loc[df['state'] == state, var].dropna()
    return x.mean(), x.std() / np.sqrt(len(x)), len(x)

def t_stat(df, var):
    nj = df.loc[df['state'] == 1, var].dropna()
    pa = df.loc[df['state'] == 0, var].dropna()
    nj_se = nj.std() / np.sqrt(len(nj))
    pa_se = pa.std() / np.sqrt(len(pa))
    diff = nj.mean() - pa.mean()
    se = np.sqrt(nj_se**2 + pa_se**2)
    return diff / se if se > 0 else np.nan

# === 1. Distribution of Store Types (percentages) ===
print("=" * 65)
print("TABLE 2 REPLICATION: Means of Key Variables")
print("=" * 65)
print(f"{'Variable':<35} {'NJ':>10} {'PA':>10} {'t':>8}")
print("-" * 65)
print("\n1. Distribution of Store Types (percentages):")

chain_map = {1: 'Burger King', 2: 'KFC', 3: 'Roy Rogers', 4: "Wendy's"}
for code, name in chain_map.items():
    df['_tmp'] = (df['chain'] == code).astype(int) * 100
    nj_m, _, _ = calc_stats(df, '_tmp', 1)
    pa_m, _, _ = calc_stats(df, '_tmp', 0)
    t = t_stat(df, '_tmp')
    print(f"   {name:<32} {nj_m:>8.1f}   {pa_m:>8.1f}   {t:>6.1f}")

df['_tmp'] = df['co_owned'] * 100
nj_m, _, _ = calc_stats(df, '_tmp', 1)
pa_m, _, _ = calc_stats(df, '_tmp', 0)
t = t_stat(df, '_tmp')
print(f"   {'Company-owned':<32} {nj_m:>8.1f}   {pa_m:>8.1f}   {t:>6.1f}")

# === 2. Means in Wave 1 ===
print("\n2. Means in Wave 1:")

df['pct_ft'] = df['empft'] / df['fte'] * 100
df['wage_425'] = (df['wage_st'] == 4.25).astype(int) * 100
df['full_meal'] = df['psoda'] + df['pfry'] + df['pentree']
df['bonus_pct'] = df['bonus'] * 100

wave1_vars = [
    ('fte', 'FTE employment'),
    ('pct_ft', 'Percentage full-time employees'),
    ('wage_st', 'Starting wage'),
    ('wage_425', 'Wage = $4.25 (percentage)'),
    ('full_meal', 'Price of full meal'),
    ('hrsopen', 'Hours open (weekday)'),
    ('bonus_pct', 'Recruiting bonus'),
]

for var, label in wave1_vars:
    nj_m, nj_se, _ = calc_stats(df, var, 1)
    pa_m, pa_se, _ = calc_stats(df, var, 0)
    t = t_stat(df, var)
    print(f"   {label:<32} {nj_m:>8.1f}   {pa_m:>8.1f}   {t:>6.1f}")
    print(f"   {'':<32} {'('+str(round(nj_se,2))+')':>8}   {'('+str(round(pa_se,2))+')':>8}")

# === 3. Means in Wave 2 ===
print("\n3. Means in Wave 2:")

df['pct_ft2'] = df['empft2'] / df['fte2'] * 100
df['wage_425_2'] = (df['wage_st2'] == 4.25).astype(int) * 100
df['wage_505_2'] = (df['wage_st2'] == 5.05).astype(int) * 100
df['full_meal2'] = df['psoda2'] + df['pfry2'] + df['pentree2']
df['special2_pct'] = df['special2'] * 100

wave2_vars = [
    ('fte2', 'FTE employment'),
    ('pct_ft2', 'Percentage full-time employees'),
    ('wage_st2', 'Starting wage'),
    ('wage_425_2', 'Wage = $4.25 (percentage)'),
    ('wage_505_2', 'Wage = $5.05 (percentage)'),
    ('full_meal2', 'Price of full meal'),
    ('hrsopen2', 'Hours open (weekday)'),
    ('special2_pct', 'Recruiting bonus'),
]

for var, label in wave2_vars:
    nj_m, nj_se, _ = calc_stats(df, var, 1)
    pa_m, pa_se, _ = calc_stats(df, var, 0)
    t = t_stat(df, var)
    print(f"   {label:<32} {nj_m:>8.1f}   {pa_m:>8.1f}   {t:>6.1f}")
    print(f"   {'':<32} {'('+str(round(nj_se,2))+')':>8}   {'('+str(round(pa_se,2))+')':>8}")

print("-" * 65)
print("Notes: Standard errors in parentheses. t = test of equality of means.")

TABLE 2 REPLICATION: Means of Key Variables
Variable                                    NJ         PA        t
-----------------------------------------------------------------

1. Distribution of Store Types (percentages):
   Burger King                          41.1       44.3     -0.5
   KFC                                  20.5       15.2      1.2
   Roy Rogers                           24.8       21.5      0.6
   Wendy's                              13.6       19.0     -1.1
   Company-owned                        34.1       35.4     -0.2

2. Means in Wave 1:
   FTE employment                       20.4       23.3     -2.0
                                      (0.51)     (1.35)
   Percentage full-time employees       32.8       35.0     -0.7
                                      (1.33)     (2.73)
   Starting wage                         4.6        4.6     -0.4
                                      (0.02)     (0.04)
   Wage = $4.25 (percentage)            30.5       32.9     -0.4
  

In [41]:
# Manual DID Calculation
NJ_before = nj['fte'].mean()
NJ_after  = nj['fte2'].mean()
PA_before = pa['fte'].mean()
PA_after  = pa['fte2'].mean()

diff_NJ = NJ_after - NJ_before
diff_PA = PA_after - PA_before
DID     = diff_NJ - diff_PA

print(f"NJ change (after - before): {diff_NJ:.2f}")
print(f"PA change (after - before): {diff_PA:.2f}")
print(f"DID estimate:               {DID:.2f}")

NJ change (after - before): 0.59
PA change (after - before): -2.17
DID estimate:               2.75


In [42]:
# DID Regression
import statsmodels.formula.api as smf

# Convert wide to long format
before = df[['sheet', 'state', 'fte']].copy()
before['post'] = 0
before['employment'] = before['fte']

after = df[['sheet', 'state', 'fte2']].copy()
after['post'] = 1
after['employment'] = after['fte2']

df_long = pd.concat([before, after], ignore_index=True)
df_long['treat'] = df_long['state']  # 1=NJ, 0=PA

print(df_long.shape)
df_long.head()

# Drop rows with missing employment
df_long = df_long.dropna(subset=['employment'])

# DID Regression with clustered standard errors
model = smf.ols('employment ~ treat + post + treat:post', data=df_long)
results = model.fit(cov_type='cluster', cov_kwds={'groups': df_long['sheet']},use_t=True)

print(results.summary())

(820, 7)
                            OLS Regression Results                            
Dep. Variable:             employment   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.806
Date:                Fri, 27 Feb 2026   Prob (F-statistic):              0.146
Time:                        23:02:17   Log-Likelihood:                -2904.2
No. Observations:                 794   AIC:                             5816.
Df Residuals:                     790   BIC:                             5835.
Df Model:                           3                                         
Covariance Type:              cluster                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.3312      1.347     17.327

In [43]:
# Summary of Results
print("=== Replication Results Summary ===")
print(f"Manual DID estimate:      {DID:.4f}")
print(f"Regression DID estimate:  {results.params['treat:post']:.4f}")
print(f"P-value:                  {results.pvalues['treat:post']:.4f}")
print("\nConclusion: The minimum wage increase did NOT reduce employment.")
print("NJ restaurants increased FTE by ~2.75 relative to PA (p<0.05).")

=== Replication Results Summary ===
Manual DID estimate:      2.7536
Regression DID estimate:  2.7536
P-value:                  0.0357

Conclusion: The minimum wage increase did NOT reduce employment.
NJ restaurants increased FTE by ~2.75 relative to PA (p<0.05).
